# Feature Engineering & Data Preprocessing
### Internship Track: Week 3 – Task 2
**Author:** Sayantani Adhikary  
**Objective:** Transform raw customer churn data into model-ready features using domain-specific feature creation, categorical encoding techniques, numerical scaling methods, and statistical feature selection.


## 1. Import Libraries & Load Raw Dataset
We import Pandas, NumPy, and Scikit-Learn preprocessing and feature selection utilities.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

sns.set_theme(style="whitegrid")
df = pd.read_csv('../data/raw_dataset.csv')
print(f"Raw Dataset Shape: {df.shape}")
df.head()


## 2. Feature Creation (Domain-Specific Engineering)
We create three meaningful features from existing attributes:
1. `AvgMonthlySpend`: `TotalCharges / TenureMonths`
2. `HighTicketRiskFlag`: Binary flag (`1` if `SupportTickets >= 3`, else `0`)
3. `SpendPerAgeRatio`: `MonthlyCharges / Age`


In [1]:
df['AvgMonthlySpend'] = (df['TotalCharges'] / df['TenureMonths']).round(2)
df['HighTicketRiskFlag'] = (df['SupportTickets'] >= 3).astype(int)
df['SpendPerAgeRatio'] = (df['MonthlyCharges'] / df['Age']).round(2)

df[['CustomerID', 'AvgMonthlySpend', 'HighTicketRiskFlag', 'SpendPerAgeRatio']].head()


## 3. Categorical Encoding Techniques
Machine learning models require numeric inputs. We apply:
* **Ordinal Encoding**: Maps contract duration order (`Month-to-month`: 0, `One year`: 1, `Two year`: 2).
* **One-Hot Encoding**: Expands `ContractType` into binary indicator columns (`drop_first=True` to avoid multicollinearity).


In [1]:
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['ContractEncoded'] = df['ContractType'].map(contract_map)

df = pd.get_dummies(df, columns=['ContractType'], drop_first=True, dtype=int)
print("Columns after encoding:", list(df.columns))
df[['ContractEncoded', 'ContractType_One year', 'ContractType_Two year']].head()


## 4. Numerical Scaling Methods (`StandardScaler` vs. `MinMaxScaler`)
Features like `TotalCharges` (104–5730) and `SupportTickets` (0–6) exist on very different scales.
We compare **Min-Max Normalization** (scales to `[0, 1]`) and **Standardization** (zero mean, unit variance).


In [1]:
num_cols = ['Age', 'MonthlyCharges', 'TotalCharges', 'TenureMonths', 'SupportTickets', 'AvgMonthlySpend', 'SpendPerAgeRatio']

# Method 1: MinMaxScaler (for range comparison)
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df[num_cols]), columns=[c + '_MinMax' for c in num_cols])
print("MinMaxScaler range check (min/max):")
print(df_minmax.agg(['min', 'max']).round(2))

# Method 2: StandardScaler (applied to updated dataset)
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[num_cols])
for i, col in enumerate(num_cols):
    df[col + '_Scaled'] = np.round(scaled_values[:, i], 4)

df[[c + '_Scaled' for c in num_cols]].describe().round(2)


## 5. Statistical Feature Selection (`SelectKBest`)
We use ANOVA F-value (`f_classif`) to rank which features are most predictive of customer `Churn`.


In [1]:
feature_cols = [c + '_Scaled' for c in num_cols] + ['HighTicketRiskFlag', 'ContractEncoded']
X = df[feature_cols]
y = df['Churn']

selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X, y)

scores_df = pd.DataFrame({
    'Feature': feature_cols,
    'F_Score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values(by='F_Score', ascending=False)

plt.figure(figsize=(9, 4.5))
sns.barplot(x='F_Score', y='Feature', data=scores_df, palette='viridis')
plt.title('Feature Selection: ANOVA F-Score Ranking', fontsize=12, fontweight='bold')
plt.xlabel('F-Score (Higher = Stronger Churn Predictor)')
plt.tight_layout()
plt.show()

print("Top 5 Selected Features:", list(X.columns[selector.get_support()]))


## 6. Export Updated Dataset
Drop the non-predictive `CustomerID` and export the model-ready dataset to `data/updated_dataset.csv`.


In [1]:
updated_df = df.drop(columns=['CustomerID'])
updated_df.to_csv('../data/updated_dataset.csv', index=False)
print("Successfully exported model-ready dataset to data/updated_dataset.csv!")
